# K2 Customer Cleansing — `PERSON` (UFUND เดิม)

**เป้าหมาย:** ทำความสะอาดและเช็กซ้ำภายในระบบด้วยกฎชุดเดียวกับ CRM / ITEC-Rep ([`scripts/ssot/cleanse.py`](../ssot/cleanse.py)) → จำนวนลูกค้าจริงที่ไม่ซ้ำ + คีย์ที่พร้อมใช้ map ข้ามระบบ

| ขั้น | ทำอะไร |
|---|---|
| Extract | ดึงทั้งตาราง (ขนาดเล็ก) · เลือกเฉพาะคอลัมน์ที่ใช้ |
| Name Standardization | ตัดคำนำหน้า · แยกชื่อ-นามสกุล · ชื่อไทยและอังกฤษจากคอลัมน์ของตัวเอง |
| Contact Validation | เบอร์ 2 เบอร์ · อีเมล · วันเกิด · เลขบัตร + checksum |
| Record Classification | ไม่มีข้อมูลระบุตัว · บัญชีร้าน · นิติบุคคล · ชื่อปลอม · เบอร์/อีเมลกลาง |
| Duplicate Matching | R0 รหัสเดียวกัน · R1 เลขบัตร · R2–R5 ชื่อ TH/EN + เบอร์/อีเมล · R7 ชื่อ + วันเกิด |
| Quality check | กลุ่มที่มีเลขบัตร ≥ 2 เลข (สัญญาณรวมผิด/เลขบัตรพิมพ์ผิด) · review queue |

**ความปลอดภัย:** read-only · credential จาก env `K2_USER` / `K2_PWD` · ค่าที่ใช้จับคู่เป็น hash ในหน่วยความจำ · ตัวอย่างทุกตัว mask · ผลรันอยู่ใน `_out/` (ไม่ขึ้น git)

In [ ]:
import sys, json, time, collections, os, re, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
sys.path.insert(0, r'C:\Projects\my-first-project\scripts')
import db
from ssot.cleanse import *
pd.set_option('future.infer_string', False)
OUT = r'C:\Projects\my-first-project\scripts\k2\_out'
cn = db.connect('k2', timeout=900); cur = cn.cursor()
print('connected')

## สิ่งที่ระบบนี้ต่างจาก CRM / ITEC

| เรื่อง | ข้อเท็จจริง | ผลต่อการ clean |
|---|---|---|
| `PERSON_ID` | **สร้างใหม่ทุกใบคำขอ** — คนเดียวยื่นหลายครั้ง = หลายแถว | ไม่มี R0 · ใช้เลขบัตรรวม (R1) |
| `TAX_ID` | เลขบัตร 13 หลัก มีเกือบทุกแถว | R1 เป็นกฎหลัก |
| ชื่อ | ไทย `FIRST_NAME`/`LAST_NAME` + อังกฤษ `FIRST_NAME_ENG`/`LAST_NAME_ENG` · คำนำหน้า `PREFIX` | 1 แถวมีทั้ง 2 ภาษา → ไม่ต้องใช้ R6 เดาข้ามภาษา |
| เบอร์ | `PHONE` + `PHONE_SECOND` | ใช้ทั้ง 2 เบอร์จับคู่ |
| อีเมล · วันเกิด | `EMAIL` · `BIRTHDAY` | R3/R5 · R7 |

## Extract

In [ ]:
t0 = time.time()
cur.execute("""SELECT PERSON_ID, PREFIX, FIRST_NAME, LAST_NAME, FIRST_NAME_ENG, LAST_NAME_ENG, TAX_ID, BIRTHDAY, PHONE, PHONE_SECOND, EMAIL, CREATE_DATE
FROM dbo.PERSON WITH (NOLOCK)""")
raw = pd.DataFrame.from_records(cur.fetchall(), columns=[c[0] for c in cur.description])
print(f'{len(raw):,} rows · {time.time() - t0:,.0f}s')
S = pd.DataFrame({'src_id': raw.PERSON_ID.astype(str), 'title': raw.PREFIX, 'first_th': raw.FIRST_NAME, 'last_th': raw.LAST_NAME,
                  'first_en': raw.FIRST_NAME_ENG, 'last_en': raw.LAST_NAME_ENG, 'idno': raw.TAX_ID, 'birth': raw.BIRTHDAY,
                  'phone1': raw.PHONE, 'phone2': raw.PHONE_SECOND, 'email': raw.EMAIL, 'link': None})
print(f'{len(S):,} customer rows')

## Standardize · Validate · Classify
- ชื่อไทยและอังกฤษทำความสะอาดแยกกัน · **ชื่อหลัก** = ไทยถ้าใช้ได้ ไม่งั้นใช้อังกฤษ
- ชื่อปลอม = ทั้งไทยและอังกฤษใช้ไม่ได้
- สถานะ: `0` บุคคล · `1` ไม่มีข้อมูลระบุตัว · `2` นิติบุคคล · `3` ชื่อปลอม · `4` บัญชีร้าน

In [ ]:
nt = clean_names(S.first_th, S.last_th, S.title.fillna('').astype(str))
ne = clean_names(S.first_en, S.last_en)
th_ok = (nt['junk'] == '').to_numpy(); en_ok = (ne['junk'] == '').to_numpy()
p1 = clean_phone(S.phone1); p2 = clean_phone(S.phone2); em = clean_email(S.email); bd = clean_birth(S.birth)
idd, idv, idk = thai_id(S.idno)
has_id = idk == 1
junk = ~th_ok & ~en_ok
contact = (p1['status'] == 'ok').to_numpy() | (p2['status'] == 'ok').to_numpy() | (em['status'] == 'ok').to_numpy()
jur = nt['juristic'].to_numpy() | ne['juristic'].to_numpy() | (idk == 2)
intl = nt['internal'].to_numpy() | ne['internal'].to_numpy()
status = np.select([junk & ~contact & ~has_id, intl, jur, junk], [1, 4, 2, 3], 0).astype('u1')
# ชื่อแยกตามภาษา (ช่องไทยที่กรอกเป็นอังกฤษ → นับเป็นชื่ออังกฤษ)
h_th = name_hash(nt['key'], ~th_ok); h_en = name_hash(ne['key'], ~en_ok)
N_th = np.where(th_ok & (nt['lang'] == 1), h_th, 0).astype('u8')
N_en = np.where(en_ok & (ne['lang'] == 2), h_en, np.where(th_ok & (nt['lang'] == 2), h_th, 0)).astype('u8')
N_any = np.where(N_th != 0, N_th, N_en).astype('u8')
a = dict(status=status, name=N_any, phone=p1['val'], email=em['hash'], company=em['company'].to_numpy())
P2 = p2['val'].copy(); P2[P2 == a['phone']] = 0
cnt = collections.Counter(rows=len(S))
for k, v in zip(*np.unique(status, return_counts=True)): cnt[f'status_{k}'] += int(v)
cnt.update(title=int(nt['title_in'].sum()), split=int(nt['split'].sum()), last_placeholder=int(nt['last_placeholder'].sum()),
           th_name_ok=int(th_ok.sum()), en_name_ok=int(en_ok.sum()), both_names=int((N_th != 0).sum() and ((N_th != 0) & (N_en != 0)).sum()),
           th_field_in_english=int((th_ok & (nt['lang'] == 2)).sum()),
           phone1_ok=int((p1['status'] == 'ok').sum()), phone2_ok=int((P2 != 0).sum()), mobile_any=int((p1['mobile'] | p2['mobile']).sum()),
           phone_fixed=int(p1['fixed'].sum() + p2['fixed'].sum()), phone_invalid=int((p1['status'] == 'invalid').sum()), phone_empty=int((p1['status'] == 'empty').sum()),
           email_ok=int((em['status'] == 'ok').sum()), email_domain_fixed=int(em['fixed_domain'].sum()), email_company=int(em['company'].sum()),
           birth_present=int((bd['val'] > 0).sum()), birth_be_fixed=int(bd['be_fixed'].sum()), birth_jan1=int(bd['jan1'].sum()), birth_invalid=int(bd['invalid'].sum()),
           id_present=int((idd != '').sum()), id_valid=int(idv.sum()), id_person=int(has_id.sum()), id_juristic=int((idk == 2).sum()),
           juristic_by_name=int(((nt['juristic'] | ne['juristic']).to_numpy() & (status == 2)).sum()))
for k, v in nt['junk'][nt['junk'] != ''].value_counts().items(): cnt[f'junk_th_{k}'] += int(v)
print({k: v for k, v in cnt.items()})

In [ ]:
# ตัวอย่าง (mask) — ข้อมูลจริงถูกปิดบัง
ex = []
for lab_, m in [('ตัดคำนำหน้า', nt['title_in']), ('แยกชื่อ-นามสกุลจากช่องเดียว', nt['split']), ('ช่องชื่อไทยกรอกเป็นอังกฤษ', pd.Series(th_ok & (nt['lang'] == 2)))]:
    for i in np.flatnonzero(m.to_numpy())[:5]:
        ex.append((lab_, mask_tok(str(S.first_th.iat[i])) + ' | ' + mask_tok(str(S.last_th.iat[i])), mask_tok(nt['first'].iat[i]) + ' | ' + mask_tok(nt['last'].iat[i])))
for i in np.flatnonzero(status == 2)[:5]:
    ex.append(('นิติบุคคล', mask_org(str(S.first_th.iat[i]) + ' ' + str(S.last_th.iat[i])), '→ ชุดนิติบุคคล'))
pd.DataFrame(ex, columns=['กฎ', 'ก่อน', 'หลัง'])

## Shared contacts + Duplicate matching
เบอร์/อีเมลเดียวที่มีชื่อต่างกัน ≥ 5 ชื่อ (เบอร์ร้าน/พนักงาน) → ไม่ใช้เป็นคีย์ · ระบบนี้มีชื่อไทยและอังกฤษในแถวเดียวกัน จึง**ไม่ใช้ R6 (เดาข้ามภาษา)**

In [ ]:
person = status == 0
Pall = np.concatenate([a['phone'], P2]); Nall = np.concatenate([N_any, N_any]); Mall = np.concatenate([person, person])
dp = distinct_per_key(Pall, Nall, Mall); shp = (dp[:len(S)] >= 5), (dp[len(S):] >= 5)
de = distinct_per_key(a['email'], N_any, person); she = de >= 5
cnt.update(shared_phone_rows=int(shp[0].sum() + shp[1].sum()), shared_email_rows=int(she.sum()))
P1m = np.where(shp[0], 0, a['phone']).astype('i8'); P2m = np.where(shp[1], 0, P2).astype('i8'); Em = np.where(she, 0, a['email']).astype('u8')
B = bd['val']; ID = id_hash(idd, has_id)
LK = np.zeros(len(S), 'u8') if S.link.isna().all() else pd.util.hash_array(S.link.fillna('').astype(str).to_numpy(object)).astype('u8')
z = np.zeros(len(S), bool)
rules = []
if not S.link.isna().all(): rules.append(('R0 รหัสลูกค้าเดียวกัน', [(LK, LK != 0)]))
rules += [('R1 เลขบัตร', [(ID, ID != 0)]),
          ('R2 ชื่อไทย+เบอร์', [(mix(N_th, P1m), (N_th != 0) & (P1m != 0)), (mix(N_th, P2m), (N_th != 0) & (P2m != 0))]),
          ('R3 ชื่อไทย+อีเมล', [(mix(N_th, Em), (N_th != 0) & (Em != 0))]),
          ('R4 ชื่ออังกฤษ+เบอร์', [(mix(N_en, P1m), (N_en != 0) & (P1m != 0)), (mix(N_en, P2m), (N_en != 0) & (P2m != 0))]),
          ('R5 ชื่ออังกฤษ+อีเมล', [(mix(N_en, Em), (N_en != 0) & (Em != 0))]),
          ('R7 ชื่อ+วันเกิด', [(mix(N_th, B), (N_th != 0) & (B != 0)), (mix(N_en, B), (N_en != 0) & (B != 0))])]
steps, n_person, unique, lab = dedup_keys(len(S), person, rules)
a['cluster'] = lab
pd.DataFrame(steps).style.format({'edges': '{:,}', 'merged': '{:,}', 'unique_after': '{:,}'})

## Quality check
- **กลุ่มที่มีเลขบัตร ≥ 2 เลข** = รวมคนต่างเลขบัตรเข้าด้วยกันผ่านชื่อ+เบอร์ (อาจเป็นเลขบัตรพิมพ์ผิด หรือรวมผิดคน) → ใช้วัดความแม่นของกฎ R2–R7
- **เลขบัตรเดียวกันแต่ชื่อต่างกัน** = เปลี่ยนชื่อ/สะกดต่าง หรือเลขบัตรผิด
- **Review queue** = ชื่อเดียวกันแต่ไม่มีคีย์ยืนยัน → ไม่รวม

In [ ]:
lab1 = lab.astype('u8') + 1
multi_id = distinct_per_key(lab1, ID, person & (ID != 0))
id_names = distinct_per_key(ID, N_any, person & (ID != 0) & (N_any != 0))
QC = dict(clusters_with_2plus_ids=int(len(np.unique(lab1[multi_id >= 2]))),
          ids_with_2plus_names=int(len(np.unique(ID[id_names >= 2]))))
review = review_same_name(a)
def any_c(flag):
    return cluster_any(a, flag)
RD = {'national_id': any_c(ID != 0), 'mobile': any_c((p1['mobile'] & (P1m != 0)) | (p2['mobile'] & (P2m != 0))), 'email': any_c(Em != 0),
      'birth': any_c(B != 0), 'th_and_en_name': any_c((N_th != 0) & (N_en != 0)), 'any_key': any_c((ID != 0) | (P1m != 0) | (P2m != 0) | (Em != 0))}
display(pd.Series(QC).to_frame('count').style.format('{:,.0f}'))
display(pd.Series(review).to_frame('review').style.format('{:,.0f}'))
pd.DataFrame({'customers': RD, '%': {k: v / unique * 100 for k, v in RD.items()}}).style.format({'customers': '{:,.0f}', '%': '{:.1f}'})

## Funnel + summary

In [ ]:
f = [('แถวดิบ', cnt['rows']), ('✂ ไม่มีข้อมูลระบุตัว', -cnt.get('status_1', 0)), ('✂ บัญชีร้าน/สาขา', -cnt.get('status_4', 0)),
     ('✂ นิติบุคคล (แยกชุด)', -cnt.get('status_2', 0)), ('✂ ชื่อปลอม/placeholder', -cnt.get('status_3', 0))]
f += [('✂ ' + s['rule'], -s['merged']) for s in steps] + [('= ลูกค้าบุคคลไม่ซ้ำ', unique)]
for l_, v in f: print(f'   {l_:<34}{v:>12,}')
SUMMARY = {'K2': dict(name='K2 PERSON · ผู้กู้ UFUND (เดิม)', funnel=f, cnt=dict(cnt), steps=steps, review=review, n_person=n_person, unique=unique,
                       readiness=RD, qc=QC), '_run': time.strftime('%Y-%m-%d %H:%M')}
os.makedirs(OUT, exist_ok=True)
json.dump(SUMMARY, open(os.path.join(OUT, 'k2_cleansing_summary.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=int)
print('saved summary (ตัวเลขรวมเท่านั้น)')

## หมายเหตุ
- เลขบัตรใน SSOT ต้อง HMAC ก่อนเก็บ (ในโน้ตบุ๊กนี้เป็น hash ในหน่วยความจำ)
- คู่ชื่อไทย-อังกฤษของคนเดียวกันใช้เป็นชุดทดสอบ transliteration สำหรับกฎ R6 ของ CRM / ITEC
- ขั้นถัดไป: map ข้ามระบบด้วยเลขบัตร (K2 ↔ ITOS ↔ ITEC-Rep) แล้วตามด้วยมือถือ+ชื่อ